# Experiment 02: Model Training & Architecture Benchmark

This notebook benchmarks:
1. **CustomGlyphCNN**: 4-Stage Conv-BN-Mish-SE (native $32\times 32$ / $64\times 64$).
2. **AdaptedResNet18**: ResNet-18 with modified stem and ImageNet weights.
3. **AdaptedMobileNetV3**: Inverted residual bottlenecks with SE attention.

All models use:
- **Class-Balanced Focal Loss** + Label Smoothing (combats 1-to-5025 sample imbalance).
- **AdamW Optimizer** with Warmup + Cosine Annealing learning rate schedule.
- **Mixed Precision (AMP)** for high-speed, memory-efficient GPU execution.

In [ ]:
import sys
from pathlib import Path
import torch

src_dir = (Path.cwd() / '..' / 'src').resolve()
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dataset import create_dataloaders, get_class_weights
from transforms import get_train_transform, get_val_transform
from models import build_model
from losses import build_loss_fn
from trainer import build_optimizer, build_scheduler, ThaiCharacterTrainer
from evaluate import plot_training_history

DATASET_DIR = (Path.cwd() / '..' / '..' / '..' / 'ThaiCharacter Dataset' / 'round2').resolve()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 1. Prepare Dataloaders with Balanced Sampling & Letterboxing

In [ ]:
BATCH_SIZE = 64
TARGET_SIZE = (32, 32)

train_loader, test_loader, train_df, test_df, class_to_idx = create_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    target_size=TARGET_SIZE,
    train_transform=get_train_transform(max_angle=8.0, p_morphology=0.3),
    test_transform=get_val_transform(),
    use_balanced_sampler=True,
    num_workers=2,
)

class_weights = get_class_weights(train_df, num_classes=len(class_to_idx), beta=0.999)
print(f'Train Batches: {len(train_loader)} | Test Batches: {len(test_loader)}')


## 2. Train CustomGlyphCNN (Baseline Model)

In [ ]:
custom_model = build_model('custom_cnn', num_classes=len(class_to_idx), dropout_rate=0.3)
criterion = build_loss_fn('class_balanced_focal', class_weights=class_weights, gamma=2.0, label_smoothing=0.08)
optimizer = build_optimizer(custom_model, optimizer_name='adamw', base_lr=1e-3, weight_decay=1e-2)
scheduler = build_scheduler(optimizer, scheduler_type='cosine_warmup', total_epochs=20, warmup_epochs=3)

trainer = ThaiCharacterTrainer(
    model=custom_model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    save_dir='../checkpoints/custom_cnn',
)

history_custom = trainer.fit(num_epochs=15)
plot_training_history(history_custom)


## 3. Train Adapted ResNet-18 (Transfer Learning with Differential LR)

In [ ]:
resnet_model = build_model('resnet18', num_classes=len(class_to_idx), pretrained=True, adapt_stem=True)
optimizer_resnet = build_optimizer(
    resnet_model,
    optimizer_name='adamw',
    base_lr=1e-3,
    differential_lr=True,
    backbone_lr=1e-4,
    weight_decay=1e-2,
)
scheduler_resnet = build_scheduler(optimizer_resnet, scheduler_type='cosine_warmup', total_epochs=20, warmup_epochs=3)

trainer_resnet = ThaiCharacterTrainer(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer_resnet,
    scheduler=scheduler_resnet,
    device=device,
    save_dir='../checkpoints/resnet18',
)

history_resnet = trainer_resnet.fit(num_epochs=15)
plot_training_history(history_resnet)
